In [ ]:
# @title Lấy city từ trong wiki database
import requests
import pandas as pd
import json
url = "https://query.wikidata.org/sparql"

query = """
SELECT ?city ?cityLabel ?countryLabel WHERE {
  ?city wdt:P31 wd:Q515.
  ?city wdt:P17 ?country.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 10
"""

headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
}

response = requests.get(url, params={"query": query}, headers=headers)

print(response.status_code)
print(response.text[:500])  # xem thử nó trả gì

data = response.json()
def get_cities_batch(limit=5000, offset=0):
    query = f"""
    SELECT ?city ?cityLabel ?countryLabel WHERE {{
      ?city wdt:P31 wd:Q515.
      ?city wdt:P17 ?country.
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    LIMIT {limit}
    OFFSET {offset}
    """

    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    r = requests.get(url, params={"query": query}, headers=headers)
    return r.json()

if response.status_code != 200:
    print("Error:", response.status_code)
    print(response.text)
else:
    try:
        data = response.json()
    except Exception as e:
        print("Not JSON response")
        print(response.text[:500])

cities = []

for item in data["results"]["bindings"]:
    cities.append({
        "city": item["cityLabel"]["value"],
        "country": item["countryLabel"]["value"],
        "wikidata_url": item["city"]["value"]
    })

len(cities)
with open("all_cities.json", "w", encoding="utf-8") as f:
    json.dump(cities, f, ensure_ascii=False, indent=2)

200
{
  "head" : {
    "vars" : [ "city", "cityLabel", "countryLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "city" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q60"
      },
      "cityLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "New York City"
      },
      "countryLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "United States"
      }
    }, {
      "city" : {
        "type" : 


In [ ]:
import pandas as pd

df = pd.DataFrame(cities[:10])
df.head()

,city,country,wikidata_url
0,New York City,United States,http://www.wikidata.org/entity/Q60
1,Los Angeles,United States,http://www.wikidata.org/entity/Q65
2,Bern,Switzerland,http://www.wikidata.org/entity/Q70
3,London,United Kingdom,http://www.wikidata.org/entity/Q84
4,Alexandria,Egypt,http://www.wikidata.org/entity/Q87


In [ ]:
import requests

def get_wikipedia_data(title):
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}"

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    r = requests.get(url, headers=headers)

    if r.status_code == 200:
        return r.json()
    else:
        return None

test = get_wikipedia_data("New York City")
print(test["extract"])
print(test.get("thumbnail"))

New York, often called New York City (NYC), is the most populous city in the United States. It is located at the southern tip of New York State on New York Harbor, one of the world's largest natural harbors. The city comprises five boroughs, each coextensive with its respective county. It is the geographical and demographic center of both the Northeast megalopolis and the New York metropolitan area, the largest metropolitan area in the United States by both population and urban area. New York is a global center of finance and commerce, culture, technology, entertainment and media, academics and scientific output, the arts and fashion, and, as home to the headquarters of the United Nations, international diplomacy.
{'source': 'https://upload.wikimedia.org/wikipedia/commons/thumb/7/7a/View_of_Empire_State_Building_from_Rockefeller_Center_New_York_City_dllu_%28cropped%29.jpg/330px-View_of_Empire_State_Building_from_Rockefeller_Center_New_York_City_dllu_%28cropped%29.jpg', 'width': 330, 'h

In [ ]:
from tqdm import tqdm
import time

results = []

for city in tqdm(df["city"]):
    data = get_wikipedia_data(city)

    if data:
        results.append({
            "title": data.get("title"),
            "text": data.get("extract"),
            "image": data.get("thumbnail", {}).get("source"),
            "url": data.get("content_urls", {}).get("desktop", {}).get("page")
        })

    time.sleep(0.2)  # tránh rate limit

len(results)

100%|██████████| 10/10 [00:03<00:00,  3.32it/s]


10

In [ ]:
import json

with open("cities.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

In [ ]:
import requests

def get_full_page_data(title):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts|images",
        "explaintext": True,
        "titles": title,
        "imlimit": "max"
    }

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    r = requests.get(url, params=params, headers=headers)
    data = r.json()

    pages = data["query"]["pages"]
    page = next(iter(pages.values()))

    text = page.get("extract", "")

    image_titles = []
    if "images" in page:
        for img in page["images"]:
            if img["title"].lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                image_titles.append(img["title"])

    return text, image_titles

In [ ]:
import requests
import time
def chunk_list(lst, size=50):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def get_image_urls(image_titles):
    if not image_titles:
        return []

    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        "User-Agent": "MyResearchBot/1.0 (your_email@example.com)"
    }

    image_urls = []

    # 🔥 chia thành batch 50
    for batch in chunk_list(image_titles, 50):

        params = {
            "action": "query",
            "format": "json",
            "prop": "imageinfo",
            "iiprop": "url",
            "titles": "|".join(batch)
        }

        r = requests.get(url, params=params, headers=headers)
        data = r.json()

        if "query" not in data:
            print("API error:", data)
            continue

        for page in data["query"]["pages"].values():
            if "imageinfo" in page:
                image_urls.append(page["imageinfo"][0]["url"])

        time.sleep(0.3)  # tránh rate limit

    return image_urls

In [ ]:
# @title Test 1 cities
title = "New York City"

text, image_titles = get_full_page_data(title)
image_urls = get_image_urls(image_titles)



In [ ]:
# @title CrawData
all_city_data = []

for city_item in tqdm(cities[:10]):
    title = city_item["city"]

    result = get_full_page_data(title)

    if result is None:
        continue
    text, image_titles = result
    image_urls = get_image_urls(image_titles)
    # if title == 'New York City':
    #   print(len(text))
    #   print(len(image_urls))
    #   break

    all_city_data.append({
        "title": title,
        "country": city_item["country"],
        "wikidata_url": city_item["wikidata_url"],
        "text": text,
        "images": image_urls,
        "wiki_url": f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
    })

    time.sleep(0.3)  # tránh bị block


100%|██████████| 10/10 [00:16<00:00,  1.62s/it]


In [ ]:
with open("all_cities_full.json", "w", encoding="utf-8") as f:
    json.dump(all_city_data, f, ensure_ascii=False, indent=2)

print("Saved:", len(all_city_data), "cities")

Saved: 10 cities


In [ ]:
import os

def get_size(start_path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

folder = "/content/drive/MyDrive/Wiki_Deep_Images"
size_gb = get_size(folder) / (1024**3)

print(f"Folder size: {size_gb:.2f} GB")

Folder size: 97.08 GB
